Week 3: Sentiment Analysis and Urgency Scoring

 Load & Prepare Data

In [11]:


import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('../data/grievances.csv')

# Clean text (same as before)
import re
def clean(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df['clean_text'] = df['text'].apply(clean)

# Encode sentiment labels to numbers
# Positive=0, Neutral=1, Negative=2, Critical/Urgent=3
le = LabelEncoder()
df['label'] = le.fit_transform(df['sentiment'])

print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))
print(df[['sentiment', 'label']].value_counts())

Label mapping: {'Critical/Urgent': np.int64(0), 'Negative': np.int64(1), 'Neutral': np.int64(2)}
sentiment        label
Negative         1        522
Critical/Urgent  0        307
Neutral          2        171
Name: count, dtype: int64


Train/Test Split

In [12]:
X = df['clean_text'].tolist()
y = df['label'].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,        # ensures minority class (Critical/Urgent) is in both splits
    random_state=42
)

print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

Train size: 800 | Test size: 200


SVM Baseline (same as before, for comparison)

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_v = tfidf.fit_transform(X_train)
X_test_v  = tfidf.transform(X_test)

svm = LinearSVC(random_state=42, max_iter=2000)
svm.fit(X_train_v, y_train)
y_pred_svm = svm.predict(X_test_v)

print("=== SVM Baseline ===")
print(classification_report(y_test, y_pred_svm,
      target_names=le.classes_))
print(f"Macro F1: {f1_score(y_test, y_pred_svm, average='macro'):.4f}")

=== SVM Baseline ===
                 precision    recall  f1-score   support

Critical/Urgent       1.00      1.00      1.00        62
       Negative       1.00      1.00      1.00       104
        Neutral       1.00      1.00      1.00        34

       accuracy                           1.00       200
      macro avg       1.00      1.00      1.00       200
   weighted avg       1.00      1.00      1.00       200

Macro F1: 1.0000


BERT Tokenization

In [14]:
# This is where BERT starts
from transformers import BertTokenizer
import torch

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize(texts, max_len=64):
    return tokenizer(
        texts,
        padding=True,           # pad shorter sentences
        truncation=True,        # cut sentences longer than max_len
        max_length=max_len,
        return_tensors='pt'     # return PyTorch tensors
    )

# Tokenize train and test sets
train_encodings = tokenize(X_train)
test_encodings  = tokenize(X_test)

print("Input IDs shape:", train_encodings['input_ids'].shape)
# Shape: (800, 64) → 800 samples, each 64 tokens long

C:\Users\Chenna Keshava\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Chenna Keshava\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Chenna Keshava\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an ad

Input IDs shape: torch.Size([800, 17])


Create PyTorch Dataset

In [15]:
from torch.utils.data import Dataset, DataLoader

class GrievanceDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = GrievanceDataset(train_encodings, y_train)
test_dataset  = GrievanceDataset(test_encodings,  y_test)

train_loader  = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader   = DataLoader(test_dataset,  batch_size=16)

print(f"Train batches: {len(train_loader)} | Test batches: {len(test_loader)}")

Train batches: 50 | Test batches: 13


 Load Pre-trained BERT & Fine-tune

In [16]:
from transformers import BertForSequenceClassification
from torch.optim import AdamW

# Detect GPU (use it if available, otherwise CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load BERT with a classification head on top
# num_labels = 4 (Positive, Neutral, Negative, Critical/Urgent)
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=4
)
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)  # small LR for fine-tuning

# --- Training Loop ---
EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} — Loss: {avg_loss:.4f}")

Using device: cpu


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/3 — Loss: 0.6997
Epoch 2/3 — Loss: 0.0909
Epoch 3/3 — Loss: 0.0262


Evaluate BERT

In [17]:
from sklearn.metrics import classification_report, f1_score

model.eval()
all_preds  = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds   = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("=== BERT Fine-tuned ===")
print(classification_report(all_labels, all_preds,
      target_names=le.classes_))
print(f"Macro F1: {f1_score(all_labels, all_preds, average='macro'):.4f}")

=== BERT Fine-tuned ===
                 precision    recall  f1-score   support

Critical/Urgent       1.00      1.00      1.00        62
       Negative       1.00      1.00      1.00       104
        Neutral       1.00      1.00      1.00        34

       accuracy                           1.00       200
      macro avg       1.00      1.00      1.00       200
   weighted avg       1.00      1.00      1.00       200

Macro F1: 1.0000


Priority Score & Compare Results

In [18]:
# Priority scoring (same logic as before, now powered by BERT)
priority_map = {
    'Positive':       1,
    'Neutral':        2,
    'Negative':       3,
    'Critical/Urgent': 5
}

def predict_sentiment_and_priority(text):
    model.eval()
    encoding = tokenize([text])
    input_ids      = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        output = model(input_ids=input_ids, attention_mask=attention_mask)

    pred_idx   = torch.argmax(output.logits, dim=1).item()
    sentiment  = le.inverse_transform([pred_idx])[0]
    priority   = priority_map[sentiment]

    return sentiment, priority

# Test on real examples
test_cases = [
    "There is no water in our area for 10 days, children are sick!",
    "The road near the park has a small bump.",
    "Thank you for fixing the electricity issue so quickly.",
    "Sewage water overflowing into homes, please send help immediately!"
]

print(f"{'Complaint':<55} {'Sentiment':<20} {'Priority'}")
print("-" * 85)
for t in test_cases:
    sent, pri = predict_sentiment_and_priority(t)
    print(f"{t[:53]:<55} {sent:<20} {pri}")

Complaint                                               Sentiment            Priority
-------------------------------------------------------------------------------------
There is no water in our area for 10 days, children a   Negative             3
The road near the park has a small bump.                Critical/Urgent      5
Thank you for fixing the electricity issue so quickly   Negative             3
Sewage water overflowing into homes, please send help   Negative             3


 Save BERT Model

In [19]:
# Save full model for Week 4 API
model.save_pretrained('../models/bert_sentiment/')
tokenizer.save_pretrained('../models/bert_sentiment/')
import pickle
with open('../models/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print("BERT model saved!")

BERT model saved!
